# Séance 9 - Le Bivarié II

Ce notebook fait suite à la séance 8 et approfondit l'analyse bi/multi-variée. Nous allons apprendre à créer des visualisations en combinant plusieurs variables et en utilisant quelques techniques avancées.

## Objectifs
- Créer des visualisations avec **plusieurs variables** simultanément
- Utiliser le **facet** pour comparer des sous-groupes
- Combiner des couches de visualisation

Rappel : Nous utilisons **pandas** pour préparer les données et **Altair** pour visualiser.

## Rappel : les bases du bivarié

Dans le notebook précédent, nous avons vu comment représenter des relations entre deux variables avec :
- Le **nuage de points** (scatter plot) pour deux variables quantitatives
- Le **graphique en barres** pour comparer des catégories principalement non-ordonées
- Le **graphique linéaire** pour illustrer des tendances principalement de catégories ordonnées

Aujourd'hui, nous allons aller plus loin en ajoutant d'autres dimmensions à nos visualisations.

In [ ]:
# ===========================================
# Installation & Chargement des bibliothèques
# ===========================================
%pip install "vegafusion[embed]>=1.5.0" "vl-convert-python>=1.6.0"

import pandas as pd
import altair as alt
import warnings

warnings.filterwarnings("ignore")

# Configuration d'Altair
alt.data_transformers.enable("vegafusion")  # exécution locale
alt.data_transformers.disable_max_rows()  # dataset large

In [ ]:
# ===========================================
# Chargement de la base de données
# ===========================================
data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df_raw = pd.read_csv(data_url, compression="gzip", low_memory=False)

In [ ]:
# Sélectionner les variables d'intérêt
selected_vars = [
    "V241156",  # thermomètre Harris
    "V241157",  # thermomètre Trump
    "V241177",  # idéologie
    "V241043",  # intention de vote
    "V241465x",  # éducation
    "V241567x",  # revenu (6 catégories)
    "V241458x",  # âge
    "V241501x",  # ethnie
    "V241550",  # sexe
    "V241004",  # intérêt politique
]

df = df_raw[selected_vars].copy()

df.columns = [
    "thermo_harris",
    "thermo_trump",
    "ideology",
    "vote_int",
    "education",
    "income_cat",
    "age",
    "ethnicity",
    "sex",
    "pol_int",
]

# Création de la variable polarisation (rappel)
df["polarisation"] = abs(df["thermo_trump"] - df["thermo_harris"])

# Création de catégories d'âge
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 29, 44, 59, 74, 100],
    labels=["18-29", "30-44", "45-59", "60-74", "75+"],
)

df.head()

## 1. Les graphiques en barres (suite...)

Nous avons brièvement vu dans le notebook précédent comment ajouter une couleur pour représenter une troisième variable. Développons cette idée en l'appliquant à la polarisation.

Partons d'une **question de recherche** :

- Dans quelle mesure la polarisation politique varie-t-elle selon l’âge des répondants, et quel rôle joue l’intérêt politique dans cette variation ?

Pour répondre à cette question, nous allons :
1. Calculer la polarisation (différence absolue entre les thermomètres)
2. Créer un graphique en barres entre l'âge et la polarisation
3. Séparer les barres selon l'intérêt politique

In [ ]:
# Préparation des données
mask = (
    df["polarisation"].between(0, 100)
    & df["age_group"].notna()
    & df["pol_int"].between(1, 5)
)

vars = ["age_group", "polarisation", "pol_int"]
df_bar = df[mask][vars]
df_bar


In [ ]:
# Recodage de l'intérêt politique
# Notez que nous inversons volontairement le numéro
pol_int_labels = {
    1: "5. Toujours",
    2: "4. La plupart du temps",
    3: "3. Environ la moitié du temps",
    4: "2. Parfois",
    5: "1. Jamais",
}
df_bar["pol_int"] = df_bar["pol_int"].replace(pol_int_labels)
df_bar


In [ ]:
# Calcul des moyennes par groupe
df_pol_summary = df_bar.groupby(["age_group"], as_index=False)["polarisation"].mean()
df_pol_summary

In [ ]:
alt.Chart(
    df_pol_summary,
    title=alt.TitleParams(
        text="Relation entre âge et polarisation",
        subtitle=[
            "...",
            "Source : ANES 2024 Time Series Study",
        ],
        anchor="start",
    ),
    width=400,
    height=300,
).mark_bar().encode(
    x=alt.X(
        "age_group",
        type="ordinal",
        title="Âge du répondant",
    ),
    y=alt.Y(
        "polarisation",
        type="quantitative",
        title="Polarisation",
        scale=alt.Scale(domain=[0, 100]),
    ),
)

- Que pouvons nous dire de ce graphique? 
- Est-ce ok de faire un graphique en barre? 
- Comment pouvons nous améliorer ce graphique? 


Hack-Time
- Inversez l'axe x et y et observez ce qui se passe

## 2. Comparer des sous-groupes en utilisant les "facet"

La technique du **facet** permet de diviser un graphique en plusieurs sous-graphes selon les modalités d'une variable. C'est très utile pour comparer des groupes sans surcharger un seul graphique.

### Quand l'utiliser ?
- Pour comparer plusieurs groupes sur une même visualisation
- Pour éviter de surcharger un graphique avec trop d'informations
- Pour montrer des patterns similaires dans différents sous-groupes

**Question** : Comment la polarisation varie-t-elle selon l'âge et le niveau d'intérêt politique ?

In [ ]:
# Préparation des données pour le facet
mask = (
    df["polarisation"].between(0, 100)
    & df["age_group"].notna()
    & df["pol_int"].between(1, 5)
)

df_facet = df[mask][["age_group", "polarisation", "pol_int"]].copy()
df_facet["pol_int"] = df_facet["pol_int"].replace(pol_int_labels)

# Calcul des moyennes par groupe
pol_means_facet = df_facet.groupby(["age_group", "pol_int"], as_index=False)[
    "polarisation"
].mean()
pol_means_facet.head(10)

In [ ]:
alt.Chart(pol_means_facet).mark_bar().encode(
    x=alt.X(
        "polarisation",
        type="quantitative",
        title="Polarisation moyenne",
    ),
    y=alt.Y(
        "age_group",
        type="ordinal",
        title="",
    ),
).facet(
    "pol_int",
    #    title="Polarisation moyenne par âge et intérêt politique"
)

Notez que l'ordre des modalités est logique. Cela est possible grâce au recodage de l'intérêt politique effectué en début du notebook.
- Il y a d'autres manières de définir l'ordre. Nous le verrons dans les modules sur les finitions.

### Hack-Time

Créez un graphique similaire mais en utilisant :
- Le **thermomètre Harris** (au lieu de la polarisation)
- Le **sexe** comme facet (au lieu de l'intérêt politique)
- L'**âge** en catégories

**Questions de recherche** : Les femmes et les hommes évaluent-ils différemment Harris selon leur âge ?

In [ ]:
# Hack-Time : votre code ici


## 4. Simplifications

Comment pouvons nous simplifier un graphique?

Dans notre exemple sur l'intérêt politique, en diminuant le nombre catégories, nous pouvons simplifier le graphique.

Pour ce faire, nous pouvons créer une variable qui distingue les personnes "très intéressées" par la politique des autres. Cela nous permet de comparer plus facilement les distributions de polarisation entre ces deux groupes.

In [ ]:
# Création d'une variable fort intérêt politique dans le df initial
df_bar["pol_int_high"] = (df_bar["pol_int"] == "5. Toujours").astype(int)
df_bar["pol_int_high"] = df_bar["pol_int_high"].replace({0: "Non", 1: "Oui"})
df_bar["pol_int_high"].value_counts()

In [ ]:
# Calcul des moyennes par groupe avec la nouvelle variable
df_bar_summary = df_bar.groupby(["age_group", "pol_int_high"], as_index=False)[
    "polarisation"
].mean()
df_bar_summary

In [ ]:
alt.Chart(df_bar_summary).mark_bar().encode(
    x=alt.X(
        "polarisation",
        type="quantitative",
        title="Polarisation",
    ),
    y=alt.Y(
        "age_group",
        type="ordinal",
        title="",
    ),
    color=alt.Color(
        "pol_int_high",
        type="nominal",
        title=["Fort intérêt", "politique"],
    ),
    # yOffset=alt.YOffset(
    #     "pol_int_high",
    #     type="ordinal"
    # )
).properties(
    title=alt.TitleParams(
        text="Distribution de la polarisation par groupe d'âge et intérêt politique",
        subtitle=[
            "...",
            "Source : ANES 2024 Time Series Study",
        ],
        anchor="start",
    ),
    width=400,
    height=200,
)

### Hack-Time
- Dans le graphique ci-dessus, remarquez la partie de code commentée, décommentez la et relancer le graphique.
- De même, que se passe-t-il, dans l'échelle de couleur lorsque nous utilisons le type "ordinal"?

## 4. Graphique linéaire avec plusieurs séries

Il est souvent utile de comparer **plusieurs groupes** sur un même graphique linéaire pour mettre en évidence des différences de tendances.

**Question** : Comment l'évaluation de Harris évolue-t-elle avec l'idéologie selon le niveau d'éducation ?

In [ ]:
# Préparation des données
mask = (
    (df["ideology"].between(1, 7))
    & (df["thermo_harris"].between(0, 100))
    & (df["education"].between(1, 5))
)

df_line = df[mask][["ideology", "thermo_harris", "education"]].copy()

# Recodage de l'éducation
edu_labels = {
    1: "Sans diplôme",
    2: "Baccalauréat",
    3: "Études sup. partielles",
    4: "Licence",
    5: "Master+",
}

df_line["education"] = df_line["education"].replace(edu_labels)

# Calcul des moyennes
line_means = df_line.groupby(["ideology", "education"], as_index=False)[
    "thermo_harris"
].mean()
line_means.head(10)

In [ ]:
alt.Chart(line_means).mark_line(point=True, size=2.5).encode(
    x=alt.X(
        "ideology",
        type="ordinal",
        title="Idéologie (1 = très libéral, 7 = très conservateur)",
    ),
    y=alt.Y(
        "thermo_harris",
        type="quantitative",
        title="Thermomètre Harris (moyenne)",
        scale=alt.Scale(domain=[0, 100]),
    ),
    color=alt.Color(
        "education",
        type="nominal",
        title="Niveau d'éducation",
        sort=list(edu_labels.values()),
    ),
).properties(
    title=alt.TitleParams(
        text="Évaluation de Harris selon l'idéologie et le niveau d'éducation",
        subtitle=[
            "Les niveaux d'éducation plus élevés tendent à mieux évaluer Harris.",
            "Source : ANES 2024 Time Series Study",
        ],
        anchor="start",
    ),
    width=550,
    height=350,
)

**Interprétation** : 
- On observe une **nette baisse** de l'évaluation de Harris lorsque l'idéologie augmente (va vers la droite)
- Les personnes avec un **niveau d'éducation plus élevé** donnent généralement de meilleures évaluations à Harris

### Hack-Time

Reproduisez ce graphique mais en utilisant le **thermomètre Trump** au lieu de Harris.

Que constatez-vous ? Les tendances sont-elles inversées ?

In [ ]:
# Hack-Time : votre code ici


## 5. Combiner plusieurs couches (layering)

Altair permet de **combiner** plusieurs visualizations sur un même graphique. Cela peut être utile pour ajouter des éléments de référence (comme une ligne horizontale à 50 pour représenter la neutre).

**Exemple** : Reprenons le graphique de la polarisation par âge, et ajoutons une ligne de référence à 50 (polarisation moyenne).

In [ ]:
# Préparation des données
mask = (df["polarisation"].between(0, 100)) & df["age_group"].notna()

df_layer = df[mask][["age_group", "polarisation"]].copy()
pol_means_layer = df_layer.groupby("age_group", as_index=False)["polarisation"].mean()

# Calcul de la moyenne globale
polarisation_moyenne = df_layer["polarisation"].mean()
print(polarisation_moyenne)
pol_mean_df = pd.DataFrame({"y": [polarisation_moyenne]})
pol_mean_df

In [ ]:
# Création du graphique en couches
base = alt.Chart(pol_means_layer).encode(
    x=alt.X("age_group", type="ordinal", title=""),
)


In [ ]:
# Couche 1 : Les barres
bars = base.mark_line(color="#8B5CF6", opacity=0.7).encode(
    y=alt.Y("polarisation", type="quantitative", title="Polarisation moyenne"),
)


In [ ]:
# Couche 2 : La ligne horizontale de référence
rule = (
    alt.Chart(pol_mean_df)
    .mark_rule(
        color="red",
        strokeDash=[5, 5],
    )
    .encode(
        y=alt.Y("y", type="quantitative", title=""),
    )
)


In [ ]:
# Couche 3 : Les points pour plus de lisibilité
points = base.mark_circle(size=80, color="#8B5CF6").encode(
    y=alt.Y("polarisation", type="quantitative"),
)


In [ ]:
# Combinaison des couches
chart = (bars + points + rule).properties(
    title=alt.TitleParams(
        text="Polarisation politique par groupe d'âge",
        subtitle=[
            f"La ligne rouge pointillée représente la polarisation moyenne ({polarisation_moyenne:.2f}).",
            "Source : ANES 2024 Time Series Study",
        ],
        anchor="start",
    ),
    width=400,
    height=300,
)


In [ ]:
chart

La ligne pointillée rouge permet de visualiser rapidement que les groupes au-dessus/en-dessous de la moyenne.

## Comment choisir la "bonne" visualisation ?

Au delà des principe des base, l'essentiel est de se demander :
- **À quelle question** je veux répondre ?
- **Quelle(s) variable(s)** je veux représenter pour m'aider ?
- **Qui est mon public** ? (scientifique, spécifique, grand public)

## Hack-Time

En combinant les techniques vues aujourd'hui, créez une visualisation complète qui répond à une question de recherche de votre choix sur les données ANES 2024.

Votre graphique doit inclure :
- Un **titre explicite**
- Un **sous-titre** avec un exemple de lecture
- Une **source**
- Une **mise en forme** soignée (couleurs, proportions)

### Exemples de questions de recherche

**1. Confiance politique et vote**
- Les personnes qui font confiance au gouvernement évaluent-elles différemment les candidats ?

**2. Opinion sur l'économie et idéologie**
- L'évaluation de la situation économique personnelle varie-t-elle selon l'idéologie ?

**3. Participation politique et démographie**
- Les personnes ayant un niveau d'éducation élevé participent-elles davantage à des activités politiques ?

N'hésitez pas à réutiliser les variables déjà explorées ou à en sélectionner de nouvelles dans le codebook :
https://sda.berkeley.edu/sdaweb/docs/anes2024full/DOC/hcbkh01.htm

In [ ]:
# Hack-Time : votre code ici